# 📦 Python Dependency Management with `pip` & Requirements

> **Module:** Package Management | **Focus:** Standard Tooling & Dependency Architecture

`pip` (*"Pip Installs Packages"*) is the standard, official package manager for Python. It connects to the **Python Package Index (PyPI)** and enables developers to install, upgrade, configure, and manage third-party libraries and their dependencies cleanly.

---

## 📋 Table of Contents
1. [Introduction to `pip` & PyPI](#1.-Introduction-to-pip-&-PyPI)
2. [Core Package Operations (`install`, `upgrade`, `uninstall`)](#2.-Core-Package-Operations)
3. [Programmatic Package Introspection with `importlib.metadata`](#3.-Programmatic-Package-Introspection)
4. [Requirements Files & Semantic Versioning (SemVer)](#4.-Requirements-Files-&-Semantic-Versioning)
5. [Distribution Formats: Wheels (`.whl`) vs. Source Distributions (`sdist`)](#5.-Distribution-Formats)
6. [Custom Indexes, Git Installs & Editable Mode](#6.-Custom-Indexes,-Git-Installs-&-Editable-Mode)
7. [Supply Chain Security & Hash Verification](#7.-Supply-Chain-Security-&-Hash-Verification)
8. [Pip Configuration (`pip.ini` / `pip.conf`) & Environment Variables](#8.-Pip-Configuration)
9. [Real-World Case Studies & Deployment Workflows](#9.-Real-World-Case-Studies)
10. [Common Pitfalls & Anti-Patterns](#10.-Common-Pitfalls-&-Anti-Patterns)
11. [Hands-On Interactive Challenges](#11.-Hands-On-Interactive-Challenges)
12. [Quick Reference Card & Summary Cheat Sheet](#12.-Quick-Reference-Card-&-Summary-Cheat-Sheet)


---
## 1. Introduction to `pip` & PyPI

### 💡 Why Always Run `python -m pip`?
Running bare `pip` executes whichever `pip` executable appears first on your system `PATH`, which may belong to a completely different Python installation.
Invoking `python -m pip` guarantees that packages are installed **specifically into the active Python interpreter's environment**.


In [ ]:
import sys
import subprocess

# 1. Inspect active Python interpreter and pip version
print(f"Active Python Interpreter: {sys.executable}")
print(f"Python Version:            {sys.version.split()[0]}")

res = subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True, text=True, check=True)
print(f"Active pip Version:        {res.stdout.strip()}")

# 2. Inspect pip cache location
res_cache = subprocess.run([sys.executable, "-m", "pip", "cache", "dir"], capture_output=True, text=True)
print(f"Pip Cache Directory:       {res_cache.stdout.strip()}")


---
## 2. Core Package Operations

Common `pip` CLI commands:
- `python -m pip install <package>`: Installs the latest compatible release.
- `python -m pip install <package>==1.2.3`: Installs an exact pinned version.
- `python -m pip install --upgrade <package>` (or `-U`): Upgrades package to the newest version.
- `python -m pip uninstall -y <package>`: Uninstalls package.
- `python -m pip list`: Lists all installed packages.
- `python -m pip show <package>`: Displays package metadata, license, and dependencies.


In [ ]:
# Inspect metadata of a built-in / installed package using pip show
res_show = subprocess.run([sys.executable, "-m", "pip", "show", "pip"], capture_output=True, text=True)
print("[Metadata for 'pip' package]:")
for line in res_show.stdout.splitlines()[:8]:
    print(" ", line)


---
## 3. Programmatic Package Introspection with `importlib.metadata`

Python 3.8+ includes `importlib.metadata` in the standard library to query installed distributions programmatically without shelling out to `pip`.


In [ ]:
import importlib.metadata

# Inspect installed distributions programmatically
dists = list(importlib.metadata.distributions())
print(f"Total installed distributions in environment: {len(dists)}")

print("\nSample of installed packages:")
for d in dists[:6]:
    print(f"  -> {d.metadata['Name']:<25} (Version: {d.version})")


---
## 4. Requirements Files & Semantic Versioning (SemVer)

### 📌 Semantic Versioning (`MAJOR.MINOR.PATCH`)
- **MAJOR (`X.0.0`)**: Breaking API changes.
- **MINOR (`0.Y.0`)**: Backwards-compatible new features.
- **PATCH (`0.0.Z`)**: Backwards-compatible bug fixes.

### 📐 Version Specifiers
| Specifier | Example | Meaning |
| :--- | :--- | :--- |
| `==` | `requests==2.31.0` | Exact version match |
| `>=` / `<=` | `urllib3>=1.26.0,<3.0` | Version range bounds |
| `~=` | `fastapi~=0.100.0` | Compatible release (`>=0.100.0, ==0.*`) |
| `!=` | `pandas!=1.5.0` | Exclude buggy version |
| `;` | `pywin32; sys_platform == 'win32'` | Environment marker |


In [ ]:
sample_requirements = """# Base Application Dependencies
fastapi>=0.100.0,<1.0.0
pydantic~=2.5.0
requests==2.31.0
colorama>=0.4.6; sys_platform == 'win32'
pytest>=7.0.0 # Testing tool
"""

print("Example requirements.txt structure:")
print("-" * 50)
print(sample_requirements.strip())
print("-" * 50)


---
## 5. Distribution Formats: Wheels (`.whl`) vs. Source Distributions (`sdist`)

### 📦 Source Distribution (`sdist`, `.tar.gz` / `.zip`)
- Contains raw Python source code and C/C++ source files.
- Requires local build tools (C compiler, header files, CMake) on the target machine during installation.

### ⚡ Built Wheel (`.whl`, PEP 427)
- Pre-built distribution containing ready-to-copy files.
- Zero compile time during installation.

### 🏷️ Wheel Tag Anatomy
`{distribution}-{version}(-{build tag})?-{python tag}-{abi tag}-{platform tag}.whl`
- **Pure Python**: `requests-2.31.0-py3-none-any.whl` (runs on any OS & Python 3.x).
- **Platform-Specific (Windows x86_64)**: `numpy-1.26.0-cp311-cp311-win_amd64.whl`
- **Platform-Specific (Linux x86_64)**: `numpy-1.26.0-cp311-cp311-manylinux_2_17_x86_64.whl`


In [ ]:
def parse_wheel_filename(filename: str) -> dict[str, str] | None:
    """Parses a standard PEP 427 wheel filename into its constituent tags."""
    if not filename.endswith(".whl"):
        return None
    name_without_ext = filename[:-4]
    parts = name_without_ext.split("-")
    if len(parts) < 5:
        return None
    return {
        "distribution": parts[0],
        "version": parts[1],
        "python_tag": parts[-3],
        "abi_tag": parts[-2],
        "platform_tag": parts[-1],
        "is_pure_python": parts[-1] == "any" and parts[-2] == "none"
    }

pure_whl = parse_wheel_filename("requests-2.31.0-py3-none-any.whl")
binary_whl = parse_wheel_filename("numpy-1.26.0-cp311-cp311-win_amd64.whl")

print("Pure Python Wheel:   ", pure_whl)
print("Binary Compiled Wheel:", binary_whl)


---
## 6. Custom Indexes, Git Installs & Editable Mode

### 🌐 Private Registries & Local Directories
- `--index-url <url>`: Replaces PyPI with a private repository (e.g. AWS CodeArtifact, Nexus).
- `--extra-index-url <url>`: Adds secondary index to search in addition to PyPI.
- `--no-index --find-links <dir>`: Installs strictly from a local directory of downloaded `.whl` files (ideal for air-gapped secure servers).

### 🛠️ Git Installs & Editable Mode
- `pip install git+https://github.com/psf/requests.git@v2.31.0`: Installs directly from Git branch or tag.
- `pip install -e .`: Editable install (creates a `.pth` link to source code so local code edits take effect immediately without re-installing).


---
## 7. Supply Chain Security & Hash Verification

To prevent **dependency hijacking** or man-in-the-middle attacks, requirements files can specify cryptographic SHA-256 hashes using `--require-hashes`:

```text
requests==2.31.0 \
    --hash=sha256:58cd2187c01e70e6e26505bca751777aa9f2ee0b7f4300988b709f44e013003f \
    --hash=sha256:942c5a758f98d790eaed1a29cb6eefc7ffb0d1cf7af05c3d2791656dbd6dd1ef
```
`pip` verifies that the downloaded `.whl` matches the exact hash before executing or installing anything.


In [ ]:
import hashlib

def compute_sha256(content: bytes) -> str:
    """Computes SHA-256 hex digest for package verification."""
    return hashlib.sha256(content).hexdigest()

sample_bytes = b"print('Hello, Python Gym!')"
checksum = compute_sha256(sample_bytes)
print(f"Computed SHA-256 Hash: sha256:{checksum}")


---
## 8. Pip Configuration (`pip.ini` / `pip.conf`) & Environment Variables

### 📂 Config File Locations
- **Windows**: `%APPDATA%\pip\pip.ini`
- **Linux / macOS**: `~/.config/pip/pip.conf`

### ⚙️ Useful Environment Variables
- `PIP_INDEX_URL`: Default repository URL.
- `PIP_NO_CACHE_DIR=1`: Disables caching (useful in Docker container builds).
- `PIP_TIMEOUT=60`: Sets socket timeout for slow connections.


In [ ]:
# Query pip configuration settings
res_config = subprocess.run([sys.executable, "-m", "pip", "config", "list"], capture_output=True, text=True)
config_output = res_config.stdout.strip()
fallback_msg = "  (Using default global PyPI settings)"
print(f"Active Pip Config Settings:\n{config_output if config_output else fallback_msg}")


---
## 9. Real-World Case Studies & Deployment Workflows

### 📁 Case Study 1: Production Multi-Tier Requirements Hierarchy

```text
project_root/
├── requirements/
│   ├── base.txt       # Core application runtime dependencies
│   ├── dev.txt        # Linters, debuggers, formatters (-r base.txt)
│   └── test.txt       # Pytest, coverage, mocks (-r base.txt)
```

In `requirements/dev.txt`:
```text
-r base.txt
black>=23.0.0
ruff>=0.1.0
mypy>=1.5.0
```

### 🔒 Case Study 2: Air-Gapped / Offline Deployment
1. **On Internet-Connected Build Machine**:
   ```bash
   python -m pip download -r requirements.txt -d ./vendor_wheels
   ```
2. **On Air-Gapped Production Server**:
   ```bash
   python -m pip install --no-index --find-links=./vendor_wheels -r requirements.txt
   ```


---
## 10. Common Pitfalls & Anti-Patterns

### ❌ Pitfall 1: Unpinned Dependencies in Production
*Anti-Pattern*: Writing `requests` without version numbers. A new major release could break your production deployment overnight.
*Best Practice*: Pin exact versions (`requests==2.31.0`) or compatible release bounds (`requests~=2.31.0`).

### ❌ Pitfall 2: `pip freeze` Environment Pollution
*Anti-Pattern*: Blindly running `pip freeze > requirements.txt` on a dirty global Python environment.
*Best Practice*: Use isolated virtual environments and separate direct dependencies from transitive ones.

### ❌ Pitfall 3: Modifying System Python
*Anti-Pattern*: Running `sudo pip install` or modifying OS Python.
*Best Practice*: Always create a dedicated `venv` for each project.


---
## 11. Hands-On Interactive Challenges


In [ ]:
import re

# Challenge 1: Parse requirements file into structured dictionaries
def parse_requirements(req_content: str) -> list[dict[str, str]]:
    results = []
    for line in req_content.splitlines():
        line = line.strip()
        # Remove comments and empty lines
        if not line or line.startswith("#") or line.startswith("-"):
            continue
        # Split inline comments
        line = line.split("#")[0].strip()
        # Extract package name and version constraint
        match = re.match(r"^([A-Za-z0-9_.-]+)(.*)$", line)
        if match:
            pkg_name = match.group(1).strip()
            constraint = match.group(2).strip()
            results.append({"name": pkg_name, "constraint": constraint})
    return results

# Challenge 2: Compatible Release (~=) Rule Evaluator
def is_compatible_release(installed_version: str, constraint_version: str) -> bool:
    """Evaluates whether installed_version satisfies ~= constraint_version.
    e.g. ~= 1.4.2 matches >= 1.4.2, == 1.4.*
    """
    inst_parts = [int(x) for x in installed_version.split(".")]
    req_parts = [int(x) for x in constraint_version.split(".")]
    
    if len(inst_parts) < len(req_parts):
        return False
    # Must be >= constraint
    if inst_parts < req_parts:
        return False
    # All parts except the last part of req_parts must match exactly
    prefix_len = len(req_parts) - 1
    return inst_parts[:prefix_len] == req_parts[:prefix_len]

# Automated verification tests
sample_reqs = """
# Core requirements
fastapi>=0.100.0
pydantic~=2.5.0 # Schema validation
requests==2.31.0
"""

parsed = parse_requirements(sample_reqs)
assert len(parsed) == 3
assert parsed[0] == {"name": "fastapi", "constraint": ">=0.100.0"}
assert parsed[1] == {"name": "pydantic", "constraint": "~=2.5.0"}
assert parsed[2] == {"name": "requests", "constraint": "==2.31.0"}

# Test Compatible Release Evaluator (~= 1.4.2)
assert is_compatible_release("1.4.2", "1.4.2") is True
assert is_compatible_release("1.4.9", "1.4.2") is True
assert is_compatible_release("1.5.0", "1.4.2") is False  # Minor bumped
assert is_compatible_release("1.4.1", "1.4.2") is False  # Older patch

print("[OK] All Pip & Requirements Challenges Passed!")


---
## 12. Quick Reference Card & Summary Cheat Sheet

### 📊 Master Pip Commands & Flags

| Action | Command | Description |
| :--- | :--- | :--- |
| **Install Package** | `python -m pip install <pkg>` | Install latest release |
| **Install from File** | `python -m pip install -r req.txt` | Install list of dependencies |
| **Upgrade Package** | `python -m pip install -U <pkg>` | Upgrade package & dependencies |
| **Uninstall** | `python -m pip uninstall -y <pkg>` | Remove installed package |
| **Inspect Details** | `python -m pip show <pkg>` | View metadata, author, files |
| **Editable Install**| `python -m pip install -e .` | Development mode install |
| **Offline Download**| `python -m pip download -r req.txt -d ./dist` | Pre-download wheel binaries |
| **Air-gapped Install**| `python -m pip install --no-index --find-links=./dist -r req.txt` | Offline zero-network install |
| **Check Integrity** | `python -m pip check` | Verify installed dependency consistency |
| **Purge Cache** | `python -m pip cache purge` | Free disk space from wheel cache |
